In [1]:
import os
import numpy as np
from PIL import Image
from pytorch_fid import fid_score
from tqdm import tqdm

from qugen.main.generator.continuous_qgan_model_handler import ContinuousQGANModelHandler

In [2]:
image_folder_root_path = 'images_FID/'
n_FID_samples = 10_000

### Disclaimer:
**Make sure to run this notebook after having run the plot generation notebook (i.e., plots_paper.ipynb, patchQGAN_benchmark.ipynb) to ensure that all required samples are available.**

pytorch_fid is used for FID computation. Even though FID was integrated in the qugen package for this project, we use the original pytorch_fid package here for optimal comparability with other works and to speed up the evaluation process as it stores precomputed statistics (especially desirable for the real datasets).

## Precompute 10,000 samples and their statistics for FID evaluation:

Note that this step only needs to be done once per experiment. However, the initial computation may take a while!

In [3]:
experiment_image_stats = {

    # Real datasets:
    'real_MNIST': {
        'path_npy_samples': "apps/logistics/training_data/mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000.npy",
        'epoch': -1,
        'normalized': False,
    },
    'real_MNIST_0_1_2': {
        'path_npy_samples': "apps/logistics/training_data/mnist_0_1_2_32x32_N_18623.npy",
        'epoch': -1,
        'normalized': False,
    },
    'real_FashionMNIST': {
        'path_npy_samples': "apps/logistics/training_data/fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000.npy",
        'epoch': -1,
        'normalized': False,
        'inv_colors': True,
    },
    'real_FashionMNIST_0_1': {
        'path_npy_samples': "apps/logistics/training_data/fashion_mnist_0_1_32x32_N_12000.npy",
        'epoch': -1,
        'normalized': False,
        'inv_colors': True,
    },
    'real_SVHN': {
        'path_npy_samples': "apps/logistics/training_data/svhn_0_32x32_N_45550.npy",
        'epoch': -1,
        'normalized': False,
        'n_channels': 3,
    },

    # Generated data:
    'fake_MNIST_largest': {
        'path_npy_samples': "experiments/continuous_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_8921f0fc/images_by_mode/epoch=37000_n_per_mode=500_seed=2.npz",  # run plots_paper.ipynb first to ensure sample generation
        # 'epoch': 37000,
    },
    'fake_FashionMNIST_largest': {
        'path_npy_samples': "experiments/continuous_fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_c5f811fc/images_by_mode/epoch=47000_n_per_mode=500_seed=2.npz",  # run plots_paper.ipynb first to ensure sample generation
        #'epoch': 47000,
        'inv_colors': True,
    },
    'fake_SVHN_largest': {
        'experiment_path': "experiments/continuous_svhn_0_32x32_N_45550_minmax_qgan_ba56d178",
        'epoch': 59100,
        'n_channels': 3,
    },
    'fake_FashionMNIST_no_over': {
        'experiment_path': "experiments/continuous_fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_0533b651",
        'epoch': 20000,
        'inv_colors': True,
    },
    'fake_FashionMNIST_mid_over': {
        'path_npy_samples': "experiments/continuous_fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_e098dc80/images_by_mode/epoch=20000_n_per_mode=1000_seed=42.npz",  # run plots_paper.ipynb first to ensure sample generation
        # 'epoch': 20000,
        'inv_colors': True,
    },
    'fake_FashionMNIST_over': {
        'path_npy_samples': "experiments/continuous_fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_cbf8da2e/images_by_mode/epoch=36800_n_per_mode=1000_seed=42.npz",  # run plots_paper.ipynb first to ensure sample generation
        # 'epoch': 36800,
        'inv_colors': True,
    },
    'fake_FashionMNIST_over_0_1': {
        'path_npy_samples': "experiments/continuous_fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_cbf8da2e/images_by_mode/epoch=36800_n_per_mode=1250_seed=42.npz",  # run plots_paper.ipynb first to ensure sample generation
        # 'epoch': 36800,
        'inv_colors': True,
        'select_modes': [8, 14, 21, 25,  # t-shirts
                         19, 24, 30, 38  # trousers
                         ],  # only classes 0 (t-shirts) and 1
    },
    'fake_MNIST_0_1_2_abl_agnostic': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_32x32_N_18623_minmax_qgan_7f68e3ee",
        'epoch': 15000,
    },
    'fake_MNIST_0_1_2_abl_specific': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_32x32_N_18623_minmax_qgan_a3edbcf3",
        'epoch': 15000,
    },
    'fake_MNIST_0_1_2_single': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_32x32_N_18623_minmax_qgan_d241d4a5",
        'epoch': 15000,
    },
    'fake_MNIST_0_1_2_multi_no_tuning': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_32x32_N_18623_minmax_qgan_daa843cf",
        'epoch': 10000,
    },
    'fake_MNIST_0_1_2_multi': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_32x32_N_18623_minmax_qgan_a3edbcf3",  # same as abl_specific
        'epoch': 15000,
    },

    "fake_MNIST_depth_8": {
        'experiment_path': "experiments/continuous_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_f3be62bc",
        'epoch': 35500,
    },

    'fake_MNIST_depth_16': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_dbe9b988",
        'epoch': 30600,
    },

    'fake_MNIST_depth_32': {
        'experiment_path': "experiments/continuous_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000_minmax_qgan_c90b8717",
        'epoch': 38500,
    },

    # Generated images from PatchQGAN (Tsang et al. 2023):
    # See patchQGAN_benchmark.ipynb for image generation instructions
    'fake_MNIST_PatchQGAN': None,
    'fake_FashionMNIST_PatchQGAN': None,
}

In [4]:
def reload_qgan_and_sample(experiment_path, epoch, save_path, n_channels=1):
    model_handler = ContinuousQGANModelHandler()
    model_name = os.path.basename(experiment_path)
    print(f"Loading images for {model_name=} iteration {epoch}...")

    model_handler.reload(model_name=model_name, epoch=epoch)
    batch_size = 10
    all_samples = np.empty((n_FID_samples, 32, 32, n_channels)).squeeze()

    for i in tqdm(range(0, n_FID_samples, batch_size)):
        batch_samples = model_handler.sample(batch_size)
        all_samples[i:i+batch_size] = batch_samples.reshape(batch_size, 32, 32, n_channels).squeeze()

    np.save(save_path, all_samples)

def prepare_samples(image_folder, experiment_path=None, epoch=None, path_npy_samples=None, inv_colors=False, n_channels=1, normalized=True, **kwargs):
    """ Samples model if needed and stores them as uncompressed PNG files """
    if path_npy_samples is None:
        path_npy_samples = os.path.join(experiment_path, f"{n_FID_samples}_samples_iteration={epoch}.npy")  # default npy samples path

    # 1) Sample images:
    if not os.path.exists(path_npy_samples):
        if experiment_path is None:
            raise ValueError(f"Provided npy samples path {path_npy_samples} does not exist, and no experiment_path given to generate sample from. Probably, you need to run the plot generation notebook first to generate samples, or make sure that the real datasets are prepared and stored in apps/logistics/training_data/ (see README for more info)?")

        reload_qgan_and_sample(experiment_path, epoch, path_npy_samples, n_channels=n_channels)
    else:
        print(f"{image_folder}: Using existing samples from {path_npy_samples}")

    # Load samples
    images = np.load(path_npy_samples)
    if path_npy_samples.endswith(".npz"):  # stored by mode
        select_modes = kwargs.get('select_modes', None)
        if select_modes is not None:
            images = {k: v for k, v in images.items() if k in select_modes or int(k) in select_modes}
        images = np.array(list(images.values()))
        images = images.reshape(-1, *images.shape[2:]).squeeze()

    # shuffle:
    np.random.seed(42)
    np.random.shuffle(images)

    # Prepare samples:
    images = images[:n_FID_samples]
    n_pixels = images.size // (n_FID_samples * n_channels)
    img_size = round(np.sqrt(n_pixels))
    images = images.reshape(n_FID_samples, img_size, img_size, n_channels).squeeze()
    if normalized:  # assuming images are in [0, 1]
        images = 255 * images
    if inv_colors:
        images = 255 - images
    images = images.clip(0, 255).astype(np.uint8)

    # 2) Save images as uncompressed PNGs in image_folder:
    image_folder_path = os.path.join(image_folder_root_path, image_folder)
    try:
        os.makedirs(image_folder_path, exist_ok=False)
        for i, img_array in enumerate(images):
            # 'L' for 8-bit grayscale, 'RGB' for color images
            img = Image.fromarray((img_array).astype(np.uint8),
                                  mode=({1: 'L', 3: 'RGB'}[n_channels]))
            img.save(os.path.join(image_folder_path, f"img_{i}.png"), compress_level=0)
    except FileExistsError:
        print(f"{image_folder_path}: Image folder already exists, skipping.")


In [5]:
def compute_FID_stats(image_folder):

    stats_path = os.path.join(image_folder_root_path, f"{image_folder}.npz")
    if os.path.exists(stats_path):
        print(f"{image_folder}: Skip FID stats, already exist at {stats_path}.")
        return
    image_folder_path = os.path.join(image_folder_root_path, image_folder)

    # Analogous CLI: !python -m pytorch_fid --save-stats {image folder} {stats file}
    fid_score.save_fid_stats(paths=[image_folder_path, stats_path],
                             batch_size=50, device=None, dims=2048,  # default args
                             num_workers=os.cpu_count())

In [6]:
for experiment_key, info in experiment_image_stats.items():
    if info is None:
        print(f"{experiment_key}: No sampling info provided. Assumes that images exist already.")
    else:
        prepare_samples(experiment_key, **info)

    compute_FID_stats(experiment_key)

real_MNIST: Using existing samples from apps/logistics/training_data/mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000.npy
images_FID/real_MNIST: Image folder already exists, skipping.
real_MNIST: Skip FID stats, already exist at images_FID/real_MNIST.npz.
real_MNIST_0_1_2: Using existing samples from apps/logistics/training_data/mnist_0_1_2_32x32_N_18623.npy
images_FID/real_MNIST_0_1_2: Image folder already exists, skipping.
real_MNIST_0_1_2: Skip FID stats, already exist at images_FID/real_MNIST_0_1_2.npz.
real_FashionMNIST: Using existing samples from apps/logistics/training_data/fashion_mnist_0_1_2_3_4_5_6_7_8_9_32x32_N_60000.npy
images_FID/real_FashionMNIST: Image folder already exists, skipping.
real_FashionMNIST: Skip FID stats, already exist at images_FID/real_FashionMNIST.npz.
real_FashionMNIST_0_1: Using existing samples from apps/logistics/training_data/fashion_mnist_0_1_32x32_N_12000.npy
images_FID/real_FashionMNIST_0_1: Image folder already exists, skipping.
real_FashionMNIST_0_1: S

## Evaluate FID scores for selected experiments:
Define pairs of (generated images stats, real images stats) for FID evaluation.

In [7]:
fid_eval_pairs = {

    'MNIST_large': ("fake_MNIST_largest", "real_MNIST"),
    'FashionMNIST_large': ("fake_FashionMNIST_largest", "real_FashionMNIST"),
    'SVHN_large': ("fake_SVHN_largest", "real_SVHN"),

    'FashionMNIST_no_overmoding': ("fake_FashionMNIST_no_over", "real_FashionMNIST"),
    'FashionMNIST_mid_overmoding': ( "fake_FashionMNIST_mid_over", "real_FashionMNIST"),
    'FashionMNIST_overmoding': ("fake_FashionMNIST_over", "real_FashionMNIST"),

    'FashionMNIST_overmoding_0_1': ("fake_FashionMNIST_over_0_1", "real_FashionMNIST_0_1"),

    'MNIST_0_1_2_abl_agnostic': ("fake_MNIST_0_1_2_abl_agnostic", "real_MNIST_0_1_2"),
    'MNIST_0_1_2_abl_specific': ("fake_MNIST_0_1_2_abl_specific", "real_MNIST_0_1_2"),

    'MNIST_0_1_2_single_mode': ("fake_MNIST_0_1_2_single", "real_MNIST_0_1_2"),
    'MNIST_0_1_2_multi_no_tuning': ("fake_MNIST_0_1_2_multi_no_tuning", "real_MNIST_0_1_2"),
    'MNIST_0_1_2_multi_tuning': ("fake_MNIST_0_1_2_multi", "real_MNIST_0_1_2"),

    'MNIST_depth_8': ("fake_MNIST_depth_8", "real_MNIST"),
    'MNIST_depth_16': ("fake_MNIST_depth_16", "real_MNIST"),
    'MNIST_depth_32': ("fake_MNIST_depth_32", "real_MNIST"),

    # Results from PatchQGAN (Tsang et al. 2023):

    'PatchQGAN_MNIST': ("fake_MNIST_PatchQGAN", "real_MNIST_0_1_2"),
    'PatchQGAN_FashionMNIST': ("fake_FashionMNIST_PatchQGAN", "real_FashionMNIST_0_1"),
}


In [8]:

fid_evals = {}


for experiment_key, (gen_stats_name, real_stats_name) in fid_eval_pairs.items():
    gen_stats = os.path.join(image_folder_root_path, gen_stats_name)
    # ensure .npz extension
    if not gen_stats.endswith('.npz'):
        gen_stats += '.npz'
    real_stats = os.path.join(image_folder_root_path, real_stats_name)
    if not real_stats.endswith('.npz'):
        real_stats += '.npz'

    # Analogous CLI: !python -m pytorch_fid {gen_stats} {real_stats}
    fid_value = fid_score.calculate_fid_given_paths(paths=[gen_stats, real_stats],
                                                    batch_size=50, device=None, dims=2048)  # default args
    print(f"{experiment_key} FID: {fid_value}")
    fid_evals[experiment_key] = fid_value

MNIST_large FID: 118.26599124555793
FashionMNIST_large FID: 90.86686902210946
SVHN_large FID: 84.15531464010411
FashionMNIST_no_overmoding FID: 103.39428318399712
FashionMNIST_mid_overmoding FID: 97.25756361798722
FashionMNIST_overmoding FID: 70.03164257747832
FashionMNIST_overmoding_0_1 FID: 60.29539355358955
MNIST_0_1_2_abl_agnostic FID: 279.24027915471993
MNIST_0_1_2_abl_specific FID: 152.2199934648881
MNIST_0_1_2_single_mode FID: 216.5084470105018
MNIST_0_1_2_multi_no_tuning FID: 200.91518309299246
MNIST_0_1_2_multi_tuning FID: 152.2199934648881
MNIST_depth_8 FID: 157.29212847526003
MNIST_depth_16 FID: 138.72073884932254
MNIST_depth_32 FID: 108.9882667823502
PatchQGAN_MNIST FID: 206.5098709184375
PatchQGAN_FashionMNIST FID: 178.85663769540722
